In [ ]:
import functools
import json
import os
from pathlib import Path

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from etils import epath
import jax
import jax.numpy as jp
import mediapy as media
import mujoco
from mujoco_playground import registry
from mujoco_playground import wrapper
from mujoco_playground.config import robco_params

# Set environment variables for GPU rendering
os.environ["MUJOCO_GL"] = "egl"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

print("Imports complete!")

In [ ]:
# Configuration
ENV_NAME = "RobcoArm"
CHECKPOINT_PATH = "logs/RobcoArm-20251109-195508/checkpoints"  # Update this to your checkpoint path
EPISODE_LENGTH = 1000  # Episode length
SEED = 42
RENDER_HEIGHT = 480
RENDER_WIDTH = 640
RENDER_FPS = 50  # Frames per second for video

print(f"Environment: {ENV_NAME}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

## Configuration

Set the checkpoint path to your trained model.

In [ ]:
# Load checkpoint configuration
ckpt_path = epath.Path(CHECKPOINT_PATH).resolve()
config_file = ckpt_path / "config.json"

with open(config_file, "r") as f:
    env_cfg = json.load(f)

print("Environment configuration loaded:")
print(json.dumps(env_cfg, indent=2))

# Find the latest checkpoint
checkpoint_dirs = [d for d in ckpt_path.iterdir() if d.is_dir()]
checkpoint_dirs.sort(key=lambda x: int(x.name))
latest_checkpoint = checkpoint_dirs[-1] if checkpoint_dirs else None

if latest_checkpoint:
    print(f"\nLatest checkpoint found: {latest_checkpoint.name}")
else:
    print("\nNo checkpoints found!")

## Load Environment

In [ ]:
# Load the environment with the same configuration used during training
env_config = registry.get_default_config(ENV_NAME)
for key, value in env_cfg.items():
    if hasattr(env_config, key):
        setattr(env_config, key, value)

env = registry.load(ENV_NAME, config=env_config)
print(f"Environment '{ENV_NAME}' loaded successfully!")
print(f"Action dimension: {env.action_size}")
print(f"Observation dimension: {env.observation_size}")

## Load Trained Model

In [ ]:
# Get PPO configuration
ppo_params = robco_params.robco_ppo_config(ENV_NAME, env_cfg.get("impl", "jax"))

# Create network factory
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
)

# Load the trained model - DON'T wrap the environment, let train_fn do it
print(f"Loading model from: {latest_checkpoint}")

make_inference_fn, params, _ = ppo.train(
    environment=env,  # Pass unwrapped environment
    num_timesteps=0,  # No training, just load
    episode_length=EPISODE_LENGTH,
    action_repeat=env_cfg.get("action_repeat", 1),
    network_factory=network_factory,
    restore_checkpoint_path=latest_checkpoint,
    seed=SEED,
    wrap_env_fn=wrapper.wrap_for_brax_training,  # Let it wrap the environment
    num_eval_envs=128,  # Number of parallel evaluation environments
)

print("Model loaded successfully!")

# Now create a wrapped environment for our own rollouts
eval_env = wrapper.wrap_for_brax_training(
    env,
    episode_length=EPISODE_LENGTH,
    action_repeat=env_cfg.get("action_repeat", 1),
)

# Create inference function
inference_fn = make_inference_fn(params, deterministic=True)
jit_inference_fn = jax.jit(inference_fn)

print("Inference function compiled")

## Generate Rollouts

In [ ]:
# Define rollout function
def do_rollout(rng, state):
    """Run a complete episode and collect trajectory data."""
    empty_data = state.data.__class__(
        **{k: None for k in state.data.__annotations__}
    )
    empty_traj = state.__class__(**{k: None for k in state.__annotations__})
    empty_traj = empty_traj.replace(data=empty_data)

    def step(carry, _):
        state, rng = carry
        rng, act_key = jax.random.split(rng)
        act = jit_inference_fn(state.obs, act_key)[0]
        state = eval_env.step(state, act)
        traj_data = empty_traj.tree_replace({
            "data.qpos": state.data.qpos,
            "data.qvel": state.data.qvel,
            "data.time": state.data.time,
            "data.ctrl": state.data.ctrl,
            "data.mocap_pos": state.data.mocap_pos,
            "data.mocap_quat": state.data.mocap_quat,
            "data.xfrc_applied": state.data.xfrc_applied,
        })
        return (state, rng), traj_data

    _, traj = jax.lax.scan(step, (state, rng), None, length=EPISODE_LENGTH)
    return traj

print("Rollout function defined")

In [ ]:
# Sample frames
traj = rollout[::render_every]

# Render frames
frames = eval_env.render(
	traj,
	height=RENDER_HEIGHT,
	width=RENDER_WIDTH,
	scene_option=scene_option
)


media.show_video(frames, fps=fps)